# Phase 2 — Income Estimation

Two-stage architecture:
- **Stage 1**: LightGBM income band classifier
- **Stage 2**: Quantile regression within bands (Q25, Q50, Q75)

Training data: ~160K verified lending income records (60% SA : 40% SE)

Addresses Tweedie variance by:
1. Containing predictions within band boundaries
2. Predicting median (robust) instead of mean
3. Producing confidence intervals for BCI

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.income_estimation import IncomeEstimationPipeline
from src.utils import setup_logging, validate_features, validate_income_labels

setup_logging('INFO')

## Load Training Data (160K verified records)

In [ ]:
# train_df = pd.read_parquet('../data/processed/train_features.parquet')
# y_income = pd.read_parquet('../data/processed/train_labels.parquet')['verified_income']

# --- Sample for development ---
from data.sample.generate_sample import generate_sample_training_data
train_df, y_income = generate_sample_training_data(n=10_000)

# Validate
validate_features(train_df)
validate_income_labels(y_income)

## Train Income Estimation Pipeline

In [ ]:
pipeline = IncomeEstimationPipeline(config_path='../config/config.yaml')
pipeline.fit(train_df, y_income, segment_col='segment')

## Evaluate — Error Analysis by Segment and Band

In [ ]:
# test_df = pd.read_parquet('../data/processed/test_features.parquet')
# y_test = pd.read_parquet('../data/processed/test_labels.parquet')['verified_income']

eval_result = pipeline.evaluate(train_df, y_income, segment_col='segment')
print('\nSegment-level MAE:')
print(eval_result['segment_errors'])
print('\nBand-level MAE:')
print(eval_result['band_errors'])

In [ ]:
preds = eval_result['predictions']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_income.clip(0, 200_000), preds['income_estimate'].clip(0, 200_000), alpha=0.2, s=5)
axes[0].plot([0, 200_000], [0, 200_000], 'r--')
axes[0].set_xlabel('Actual Income (THB)')
axes[0].set_ylabel('Predicted Income (THB)')
axes[0].set_title('Actual vs Predicted Income')

# Interval width distribution
preds['income_interval_width'].clip(0, 50_000).hist(bins=50, ax=axes[1])
axes[1].set_title('Prediction Interval Width (Q75-Q25)')
axes[1].set_xlabel('THB')

plt.tight_layout()
plt.show()

## Save Model

In [ ]:
pipeline.save('../artifacts/models/income_estimation/')